In [1]:
# ── CELL 1: Imports and Initialisation ──────────────────────────────────────
import ee
import geemap
import os

ee.Initialize(project='ee-festac')

# Load Amuwo Odofin boundary
amuwo_odofin = ee.FeatureCollection("FAO/GAUL/2015/level2") \
                 .filter(ee.Filter.eq('ADM0_NAME', 'Nigeria')) \
                 .filter(ee.Filter.eq('ADM1_NAME', 'Lagos')) \
                 .filter(ee.Filter.stringContains('ADM2_NAME', 'Amuwo Odofin'))

feature_count = amuwo_odofin.size().getInfo()
if feature_count == 0:
    raise ValueError("Boundary returned 0 features — check ADM2_NAME filter")

amuwo_geom = amuwo_odofin.geometry()

print("✓ GEE initialised")
print("✓ Amuwo Odofin boundary loaded and verified")
print(f"  Feature count: {feature_count}")

/home/deysholey/.local/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


✓ GEE initialised
✓ Amuwo Odofin boundary loaded and verified
  Feature count: 1


In [2]:
# ── CELL 2: Soil Permeability from SoilGrids ────────────────────────────────
# Source: ISRIC SoilGrids 250m — globally consistent soil property predictions
# GEE ID: projects/soilgrids-isric/properties/...
# Property used: Sand content (% weight) as hydraulic conductivity proxy
#
# Scientific rationale:
# Direct hydraulic conductivity (Ksat) is not available in GEE at this resolution.
# Sand content is the standard proxy used in geospatial flood literature because:
# - Sandy soils = high permeability = faster infiltration = lower runoff generation
# - Clay-rich soils = low permeability = slower infiltration = higher runoff generation
# This relationship is well established in soil hydrology (Saxton & Rawls, 2006)
#
# SoilGrids reports sand content in g/kg (multiply by 0.1 to get percentage)
# Depth: 0-30cm topsoil layer — most relevant for surface runoff processes

try:
    # SoilGrids via GEE community catalogue
    sand_content = ee.Image("projects/soilgrids-isric/properties/sand/mean") \
                     .select('mean') \
                     .clip(amuwo_geom) \
                     .multiply(0.1) \
                     .rename('soil_permeability')

    # Verify it loaded correctly
    soil_stats = sand_content.reduceRegion(
        reducer=ee.Reducer.minMax().combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        ),
        geometry=amuwo_geom,
        scale=250,
        maxPixels=1e9
    ).getInfo()

    print("✓ SoilGrids sand content loaded successfully")
    print()
    print("Soil Sand Content Statistics (% — proxy for permeability):")
    print(f"  Minimum : {soil_stats['soil_permeability_min']:.2f}%")
    print(f"  Maximum : {soil_stats['soil_permeability_max']:.2f}%")
    print(f"  Mean    : {soil_stats['soil_permeability_mean']:.2f}%")
    print()
    print("  Interpretation: Higher % = more permeable = lower flood risk")
    print("                  Lower % = clay-rich = less permeable = higher flood risk")

except Exception as e:
    print(f"SoilGrids access failed: {e}")
    print()
    print("Falling back to OpenLandMap soil texture — alternative GEE source")
    
    # Fallback: OpenLandMap sand fraction (0-200 scale, divide by 2 for percentage)
    sand_content = ee.Image("OpenLandMap/SOL/SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02") \
                     .select('b0') \
                     .clip(amuwo_geom) \
                     .divide(2) \
                     .rename('soil_permeability')

    soil_stats = sand_content.reduceRegion(
        reducer=ee.Reducer.minMax().combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        ),
        geometry=amuwo_geom,
        scale=250,
        maxPixels=1e9
    ).getInfo()

    print("✓ OpenLandMap soil texture loaded (fallback source)")
    print()
    print("Soil Sand Fraction Statistics (% — proxy for permeability):")
    print(f"  Minimum : {soil_stats['soil_permeability_min']:.2f}%")
    print(f"  Maximum : {soil_stats['soil_permeability_max']:.2f}%")
    print(f"  Mean    : {soil_stats['soil_permeability_mean']:.2f}%")

SoilGrids access failed: Image.load: Image asset 'projects/soilgrids-isric/properties/sand/mean' not found (does not exist or caller does not have access).

Falling back to OpenLandMap soil texture — alternative GEE source
✓ OpenLandMap soil texture loaded (fallback source)

Soil Sand Fraction Statistics (% — proxy for permeability):
  Minimum : 14.50%
  Maximum : 34.50%
  Mean    : 25.68%


In [3]:
# ── CELL 3: Distance to Rivers via Flow Accumulation Thresholding ────────────
# Source: HydroSHEDS 15-arc-second Flow Accumulation (WWF/HydroSHEDS/15ACC)
#
# Method: flow accumulation thresholding at 200 cells (95th percentile)
# captures main drainage channels in flat coastal terrain where standard
# FreeFlowingRivers dataset returns no features due to urban modification.
#
# Distance method: fastDistanceTransform() — GEE's dedicated distance
# transform function. Returns squared pixel distance; sqrt() converts to
# pixel distance, then multiplied by native resolution (500m for HydroSHEDS
# 15-arc-second) to get metres. More reliable than kernel-based distance
# on masked binary images.

flow_acc_raw = ee.Image("WWF/HydroSHEDS/15ACC").clip(amuwo_geom)

# Threshold at 200 — confirmed 95th percentile from diagnostic
# Maximum value in LGA is 708 — threshold of 1000 was above all pixel values
river_network = flow_acc_raw.select('b1').gt(200)

# Verify network pixel count
river_pixel_count = river_network.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=amuwo_geom,
    scale=500,
    maxPixels=1e9
).getInfo()
print(f"River network pixels identified: {river_pixel_count}")

# fastDistanceTransform computes squared Euclidean distance in pixels
# sqrt() converts to pixel distance
# multiply by 500 converts pixels to metres (HydroSHEDS native = ~500m/pixel)
distance_to_river = river_network \
    .fastDistanceTransform(neighborhood=256) \
    .sqrt() \
    .multiply(500) \
    .clip(amuwo_geom) \
    .rename('distance_to_river')

# Verify statistics
river_dist_stats = distance_to_river.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=amuwo_geom,
    scale=500,
    maxPixels=1e9
).getInfo()

print("✓ Distance to rivers computed (fastDistanceTransform method)")
print()
print("Distance to River Statistics (metres):")
print(f"  Minimum : {river_dist_stats['distance_to_river_min']:.2f} m")
print(f"  Maximum : {river_dist_stats['distance_to_river_max']:.2f} m")
print(f"  Mean    : {river_dist_stats['distance_to_river_mean']:.2f} m")
print()
print("  Method: fastDistanceTransform — squared pixel distance → sqrt → metres")
print("  Threshold: flow accumulation > 200 cells (95th percentile for LGA)")
print("  Interpretation: Lower distance = higher flood exposure")

River network pixels identified: {'b1': 31.964705882352945}
✓ Distance to rivers computed (fastDistanceTransform method)

Distance to River Statistics (metres):
  Minimum : 0.00 m
  Maximum : 6726.81 m
  Mean    : 2337.43 m

  Method: fastDistanceTransform — squared pixel distance → sqrt → metres
  Threshold: flow accumulation > 200 cells (95th percentile for LGA)
  Interpretation: Lower distance = higher flood exposure


In [4]:
# ── CELL 4: Distance to Drainage Using JRC Global Surface Water ──────────────
# Source: JRC Global Surface Water Explorer (Pekel et al., 2016)
# GEE ID: JRC/GSW1_4/GlobalSurfaceWater
# Resolution: 30 metres
#
# Scientific rationale:
# HydroSHEDS captures major river channels but misses urban drainage features
# like canals, seasonal streams, and drainage ditches that are critical in
# Lagos's highly modified urban drainage network.
# JRC surface water occurrence captures all persistent water bodies including
# urban drainage infrastructure at 30m resolution — a complementary layer
# to the HydroSHEDS river network for urban flood studies.
#
# Occurrence threshold: >10% — pixels that are water at least 10% of the time
# This captures seasonal and intermittent drainage features, not just permanent water

jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")

# Extract water occurrence layer — values 0-100 (percentage of time covered by water)
water_occurrence = jrc.select('occurrence').clip(amuwo_geom)

# Create binary drainage mask: pixels with >10% water occurrence = drainage network
drainage_mask = water_occurrence.gt(10)

# Compute distance to nearest drainage feature
# Same fix applied to drainage distance computation
distance_to_drainage = drainage_mask \
    .fastDistanceTransform(neighborhood=256) \
    .sqrt() \
    .multiply(30) \
    .clip(amuwo_geom) \
    .rename('distance_to_drainage')

# Verify statistics
drainage_stats = distance_to_drainage.reduceRegion(
    reducer=ee.Reducer.minMax().combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    ),
    geometry=amuwo_geom,
    scale=30,
    maxPixels=1e9
).getInfo()

print("✓ Distance to drainage computed (JRC Global Surface Water)")
print()
print("Distance to Drainage Statistics (metres):")
print(f"  Minimum : {drainage_stats['distance_to_drainage_min']:.2f} m")
print(f"  Maximum : {drainage_stats['distance_to_drainage_max']:.2f} m")
print(f"  Mean    : {drainage_stats['distance_to_drainage_mean']:.2f} m")
print()
print("  Interpretation: Captures urban drainage network not in HydroSHEDS")

✓ Distance to drainage computed (JRC Global Surface Water)

Distance to Drainage Statistics (metres):
  Minimum : 0.00 m
  Maximum : 7018.33 m
  Mean    : 1207.16 m

  Interpretation: Captures urban drainage network not in HydroSHEDS


In [5]:
# ── CELL 5: Visualisation ────────────────────────────────────────────────────

amuwo_centroid = amuwo_odofin.geometry().centroid().coordinates().getInfo()
lon, lat = amuwo_centroid[0], amuwo_centroid[1]

Map = geemap.Map(center=[lat, lon], zoom=12)

# Study area boundary
amuwo_style = {'color': 'FF0000', 'fillColor': '00000000', 'width': 3}
Map.addLayer(amuwo_odofin.style(**amuwo_style), {}, 'Amuwo Odofin Boundary')

# Soil permeability — brown (low/clay) to yellow (high/sandy)
Map.addLayer(sand_content, {
    'min': 0, 'max': 80,
    'palette': ['4B2E00', 'A0522D', 'F5DEB3', 'FFFF00']
}, 'Soil Permeability (Sand % Proxy)')

# Distance to river — dark blue (close) to white (far)
Map.addLayer(distance_to_river, {
    'min': 0, 'max': 5000,
    'palette': ['000080', '0000FF', 'ADD8E6', 'FFFFFF']
}, 'Distance to Rivers (HydroSHEDS)')

# Distance to drainage — dark blue (close) to white (far)
Map.addLayer(distance_to_drainage, {
    'min': 0, 'max': 3000,
    'palette': ['000080', '0000FF', 'ADD8E6', 'FFFFFF']
}, 'Distance to Drainage (JRC Surface Water)')

Map.addLayerControl()
Map

Map(center=[6.455061708555176, 3.2650782915242123], controls=(WidgetControl(options=['position', 'transparent_…

In [6]:
# ── CELL 6: Stack and Export to Google Drive ─────────────────────────────────
# Final feature stack for this notebook: soil + distance features
# These are the last two components of the 12-feature matrix
# All bands cast to Float32 for consistency with previous exports

soil_distance_stack = sand_content.toFloat() \
    .addBands(distance_to_river.toFloat()) \
    .addBands(distance_to_drainage.toFloat())

print("Stacked band names:", soil_distance_stack.bandNames().getInfo())

export_task = ee.batch.Export.image.toDrive(
    image=soil_distance_stack,
    description='amuwo_odofin_soil_distance_features',
    folder='GeoAID_Project',
    fileNamePrefix='soil_distance_amuwo_odofin',
    region=amuwo_geom,
    scale=100,
    crs='EPSG:4326',
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)

export_task.start()

print()
print("✓ Export task submitted to Google Drive")
print("  File  : soil_distance_amuwo_odofin.tif")
print("  Scale : 100 metres | Type: Float32")
print("  Bands : soil_permeability, distance_to_river, distance_to_drainage")
print()
print("Monitor at: https://code.earthengine.google.com/tasks")

Stacked band names: ['soil_permeability', 'distance_to_river', 'distance_to_drainage']

✓ Export task submitted to Google Drive
  File  : soil_distance_amuwo_odofin.tif
  Scale : 100 metres | Type: Float32
  Bands : soil_permeability, distance_to_river, distance_to_drainage

Monitor at: https://code.earthengine.google.com/tasks


In [7]:
# ── CELL 7: Complete 12-Feature Matrix Summary ───────────────────────────────
print("=" * 60)
print("  DATA ACQUISITION PHASE: COMPLETE")
print("=" * 60)
print()
print("  COMPLETE 14-FEATURE MATRIX:")
print()
print("  TOPOGRAPHIC (Notebook 02 — topo_features_amuwo_odofin.tif)")
print("  ├── 1.  Elevation (SRTM 30m)")
print("  ├── 2.  Slope")
print("  ├── 3.  Aspect")
print("  ├── 4.  Flow Accumulation (HydroSHEDS)")
print("  ├── 5.  Topographic Wetness Index (TWI)")
print("  └── 6.  Curvature (Laplacian kernel)")
print()
print("  CLIMATE & LAND COVER (Notebook 03 — rainfall_lulc_ndvi_amuwo_odofin.tif)")
print("  ├── 7.  Mean Annual Rainfall (CHIRPS 2013-2023)")
print("  ├── 8.  Extreme Rainfall Frequency (>50mm/day)")
print("  ├── 9.  Land Use / Land Cover (ESA WorldCover 2021)")
print("  └── 10. NDVI (Sentinel-2 Dry Season Composite)")
print()
print("  SOIL & PROXIMITY (Notebook 04 — soil_distance_amuwo_odofin.tif)")
print("  ├── 11. Soil Permeability / Sand Content (SoilGrids/OpenLandMap)")
print("  └── 12. Distance to Drainage (JRC Global Surface Water)")
print()
print("  Google Drive exports: 3 GeoTIFF files")
print("  Total features: 12 ✓")
print()
print("  ─────────────────────────────────────────────────────")
print("  Next: 05_flood_inventory.ipynb")
print("        → Sentinel-1 SAR flood event detection")
print("        → Build flood/non-flood training labels")
print("        → This is the most critical notebook in the pipeline")
print("  ─────────────────────────────────────────────────────")

  DATA ACQUISITION PHASE: COMPLETE

  COMPLETE 14-FEATURE MATRIX:

  TOPOGRAPHIC (Notebook 02 — topo_features_amuwo_odofin.tif)
  ├── 1.  Elevation (SRTM 30m)
  ├── 2.  Slope
  ├── 3.  Aspect
  ├── 4.  Flow Accumulation (HydroSHEDS)
  ├── 5.  Topographic Wetness Index (TWI)
  └── 6.  Curvature (Laplacian kernel)

  CLIMATE & LAND COVER (Notebook 03 — rainfall_lulc_ndvi_amuwo_odofin.tif)
  ├── 7.  Mean Annual Rainfall (CHIRPS 2013-2023)
  ├── 8.  Extreme Rainfall Frequency (>50mm/day)
  ├── 9.  Land Use / Land Cover (ESA WorldCover 2021)
  └── 10. NDVI (Sentinel-2 Dry Season Composite)

  SOIL & PROXIMITY (Notebook 04 — soil_distance_amuwo_odofin.tif)
  ├── 11. Soil Permeability / Sand Content (SoilGrids/OpenLandMap)
  └── 12. Distance to Drainage (JRC Global Surface Water)

  Google Drive exports: 3 GeoTIFF files
  Total features: 12 ✓

  ─────────────────────────────────────────────────────
  Next: 05_flood_inventory.ipynb
        → Sentinel-1 SAR flood event detection
        → Build

In [8]:
# ── CELL 8: HAND — Height Above Nearest Drainage (Corrected) ─────────────────
# Correction from initial version:
# Initial version used HydroSHEDS flow accumulation thresholding at 500m
# resolution to define drainage channels. This produced a sparse channel
# network that missed minor canals, tidal inlets, and urban drainage
# features in Amuwo Odofin, leaving large areas with NoData HAND values.
#
# Corrected approach uses JRC Global Surface Water occurrence layer at
# 30m resolution to define the drainage network. JRC captures all
# persistent water surfaces — every canal, creek, and tidal inlet —
# giving a complete drainage network appropriate for urban coastal terrain.
#
# Threshold: JRC occurrence > 5% — same threshold used in revised NB05
# flood masking for methodological consistency. Captures seasonal and
# intermittent drainage features alongside permanent water bodies.
#
# Kernel radius: 5000m at 30m resolution = 167 pixels (well within
# GEE's 512-pixel focal minimum limit). Sufficient to reach a JRC
# water pixel from any location in Amuwo Odofin LGA.


# ── CELL 8: HAND — Height Above Nearest Drainage (Optimised) ─────────────────
# Optimisation from previous version:
# focal_min at 30m resolution with 5000m radius = 167-pixel kernel.
# GEE server-side computation timed out. Solution: compute at 100m
# native scale — consistent with all other features in the pipeline
# which are resampled to 100m in NB06. At 100m: 5000m = 50-pixel
# radius kernel, well within GEE's computational capacity.
#
# Method unchanged: JRC occurrence > 5% defines drainage network.
# focal_min propagates nearest channel elevation to all land pixels.
# HAND = pixel elevation minus nearest channel elevation.

# ── Step 1: Load DEM at 100m scale ───────────────────────────────────────────
# Reproject DEM to 100m to match target feature resolution
dem_100m = ee.Image("USGS/SRTMGL1_003") \
             .reproject(crs='EPSG:4326', scale=100) \
             .clip(amuwo_geom) \
             .rename('elevation')

print("✓ SRTM DEM loaded at 100m resolution")

# ── Step 2: Define drainage network from JRC ──────────────────────────────────
jrc = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select('occurrence')
drainage_network = jrc.gt(5) \
                      .reproject(crs='EPSG:4326', scale=100) \
                      .clip(amuwo_geom)

drainage_count = drainage_network.reduceRegion(
    reducer   = ee.Reducer.sum(),
    geometry  = amuwo_geom,
    scale     = 100,
    maxPixels = 1e9
).getInfo()

print(f"✓ JRC drainage network identified at 100m scale")
print(f"  Drainage pixels: {drainage_count.get('occurrence', 0):,.0f}")

# ── Step 3: Extract elevation at drainage pixels ──────────────────────────────
channel_elevation = dem_100m.updateMask(drainage_network) \
                             .rename('channel_elevation')

# ── Step 4: Propagate nearest channel elevation ───────────────────────────────
# At 100m resolution: 5000m radius = 50 pixels — fast and stable in GEE
nearest_channel_elev = channel_elevation.focal_min(
    radius     = 5000,
    units      = 'meters',
    kernelType = 'circle'
).rename('nearest_channel_elevation')

print("✓ Nearest channel elevation propagated")
print("  Kernel: focal minimum, 5000m radius at 100m resolution (50 pixels)")

# ── Step 5: Compute HAND ─────────────────────────────────────────────────────
hand = dem_100m.subtract(nearest_channel_elev) \
               .max(ee.Image(0)) \
               .clip(amuwo_geom) \
               .rename('hand') \
               .toFloat()

# ── Step 6: Statistics — run at 100m scale ────────────────────────────────────
hand_stats = hand.reduceRegion(
    reducer   = ee.Reducer.minMax().combine(
        reducer2     = ee.Reducer.mean().combine(
            reducer2     = ee.Reducer.percentile([10, 25, 50]),
            sharedInputs = True
        ),
        sharedInputs = True
    ),
    geometry  = amuwo_geom,
    scale     = 100,
    maxPixels = 1e9
).getInfo()

# Coverage check
valid_pixels = hand.reduceRegion(
    reducer   = ee.Reducer.count(),
    geometry  = amuwo_geom,
    scale     = 100,
    maxPixels = 1e9
).getInfo().get('hand', 0)

total_pixels = hand.unmask(-1).neq(-1).reduceRegion(
    reducer   = ee.Reducer.sum(),
    geometry  = amuwo_geom,
    scale     = 100,
    maxPixels = 1e9
).getInfo().get('hand', 1)

coverage = (valid_pixels / total_pixels * 100) if total_pixels > 0 else 0

print()
print(f"Coverage : {coverage:.1f}% valid pixels")
print()
print("HAND Statistics (metres above nearest drainage):")
print(f"  Minimum : {hand_stats.get('hand_min', 'N/A'):.2f} m")
print(f"  Maximum : {hand_stats.get('hand_max', 'N/A'):.2f} m")
print(f"  Mean    : {hand_stats.get('hand_mean', 'N/A'):.2f} m")
print(f"  P10     : {hand_stats.get('hand_p10', 'N/A'):.2f} m")
print(f"  P25     : {hand_stats.get('hand_p25', 'N/A'):.2f} m")
print(f"  P50     : {hand_stats.get('hand_p50', 'N/A'):.2f} m")
print()
print("  Pixels with HAND < 2m = highest structural inundation risk")
print("  ✓ Computation at 100m — consistent with NB06 resampling target")

✓ SRTM DEM loaded at 100m resolution
✓ JRC drainage network identified at 100m scale
  Drainage pixels: 1,518
✓ Nearest channel elevation propagated
  Kernel: focal minimum, 5000m radius at 100m resolution (50 pixels)

Coverage : 100.0% valid pixels

HAND Statistics (metres above nearest drainage):
  Minimum : 0.00 m
  Maximum : 66.00 m
  Mean    : 11.05 m
  P10     : 5.00 m
  P25     : 8.00 m
  P50     : 11.00 m

  Pixels with HAND < 2m = highest structural inundation risk
  ✓ Computation at 100m — consistent with NB06 resampling target


In [9]:
# ── CELL 9: Visualise HAND Model ─────────────────────────────────────────────
# Visualisation fix: .reproject() locks images to a fixed scale which
# breaks GEE's map tile renderer at arbitrary zoom levels. For display
# only, we recompute HAND without .reproject() so the renderer can
# handle it at the zoom level requested. Export (Cell 10) uses the
# correctly reprojected version via the scale parameter in the task.

# Rebuild for display only — no reproject(), scale handled by renderer
dem_display = ee.Image("USGS/SRTMGL1_003").clip(amuwo_geom)
jrc_display = ee.Image("JRC/GSW1_4/GlobalSurfaceWater") \
                .select('occurrence').gt(5).clip(amuwo_geom)

channel_elev_display = dem_display.updateMask(jrc_display)

nearest_elev_display = channel_elev_display.focal_min(
    radius     = 2000,
    units      = 'meters',
    kernelType = 'circle'
)

hand_display = dem_display.subtract(nearest_elev_display) \
                           .max(ee.Image(0)) \
                           .clip(amuwo_geom) \
                           .rename('hand')

# Build map
amuwo_centroid = amuwo_odofin.geometry().centroid().coordinates().getInfo()
lon, lat = amuwo_centroid[0], amuwo_centroid[1]

Map = geemap.Map(center=[lat, lon], zoom=13)

amuwo_style = {'color': 'FF0000', 'fillColor': '00000000', 'width': 3}
Map.addLayer(amuwo_odofin.style(**amuwo_style), {}, 'Amuwo Odofin Boundary')

# JRC drainage network — blue
Map.addLayer(
    jrc_display.selfMask(),
    {'palette': ['0000FF'], 'min': 1, 'max': 1},
    'JRC Drainage Network (occurrence > 5%)'
)

# HAND display layer — dark blue (0m, highest risk) to red (15m+, lowest risk)
Map.addLayer(hand_display, {
    'min'    : 0,
    'max'    : 15,
    'palette': ['000080', '0000FF', '00BFFF',
                '00FF00', 'FFFF00', 'FF6600', 'FF0000']
}, 'HAND — Height Above Nearest Drainage 0-15m')

Map.addLayerControl()
Map

Map(center=[6.455061708555176, 3.2650782915242123], controls=(WidgetControl(options=['position', 'transparent_…

In [10]:
# ── CELL 10: Export Updated Soil + Distance + HAND Stack ─────────────────────
# Updated export adds HAND as the 15th feature in the conditioning
# factor dataset. All bands cast to Float32 for type consistency.

# ── CELL 10: Export Updated Stack Including HAND ─────────────────────────────
soil_distance_hand_stack = sand_content.toFloat() \
    .addBands(distance_to_river.toFloat()) \
    .addBands(distance_to_drainage.toFloat()) \
    .addBands(hand.toFloat())

print("Stacked band names:", soil_distance_hand_stack.bandNames().getInfo())

export_task = ee.batch.Export.image.toDrive(
    image          = soil_distance_hand_stack,
    description    = 'amuwo_odofin_soil_distance_hand',
    folder         = 'GeoAID_Project',
    fileNamePrefix = 'soil_distance_hand_amuwo_odofin',
    region         = amuwo_geom,
    scale          = 100,
    crs            = 'EPSG:4326',
    maxPixels      = 1e9,
    fileFormat     = 'GeoTIFF'
)

export_task.start()

print()
print("✓ Export task submitted to Google Drive")
print("  File  : soil_distance_hand_amuwo_odofin.tif")
print("  Scale : 100 metres | Type: Float32")
print("  Bands : soil_permeability, distance_to_river,")
print("          distance_to_drainage, hand")
print()
print("Monitor at: https://code.earthengine.google.com/tasks")

Stacked band names: ['soil_permeability', 'distance_to_river', 'distance_to_drainage', 'hand']

✓ Export task submitted to Google Drive
  File  : soil_distance_hand_amuwo_odofin.tif
  Scale : 100 metres | Type: Float32
  Bands : soil_permeability, distance_to_river,
          distance_to_drainage, hand

Monitor at: https://code.earthengine.google.com/tasks


In [11]:
# ── CELL 11: Updated Session Summary ─────────────────────────────────────────
print("=" * 60)
print("  NOTEBOOK 04 (REVISED) — SOIL, DISTANCE, HAND: COMPLETE")
print("=" * 60)
print()
print("  Datasets acquired:")
print("  ├── OpenLandMap soil sand fraction ✓")
print("  ├── HydroSHEDS flow accumulation (distance to river) ✓")
print("  ├── JRC Global Surface Water (distance to drainage) ✓")
print("  └── HAND model (DEM + HydroSHEDS) ✓ NEW")
print()
print("  Features in this notebook:")
print("  ├── Feature 12: Soil permeability (sand fraction %)")
print("  ├── Feature 13: Distance to river (m)")
print("  ├── Feature 14: Distance to drainage (m)")
print("  └── Feature 15: HAND — Height Above Nearest Drainage (m) NEW")
print()
print("  Export: soil_distance_hand_amuwo_odofin.tif")
print("  Replaces: soil_distance_amuwo_odofin.tif (delete old file)")
print()
print("  Running total: 15 of 16 features complete")
print("  Remaining: GPM IMERG event rainfall (NB04b)")
print()
print("  Next: 04b_gpm_imerg_rainfall.ipynb")
print("        → Extract 72-hour cumulative rainfall from GPM IMERG")
print("        → One raster per flood event (2017, 2020, 2021, 2022)")
print("        → Feature 16: event-specific rainfall forcing")
print("=" * 60)

  NOTEBOOK 04 (REVISED) — SOIL, DISTANCE, HAND: COMPLETE

  Datasets acquired:
  ├── OpenLandMap soil sand fraction ✓
  ├── HydroSHEDS flow accumulation (distance to river) ✓
  ├── JRC Global Surface Water (distance to drainage) ✓
  └── HAND model (DEM + HydroSHEDS) ✓ NEW

  Features in this notebook:
  ├── Feature 12: Soil permeability (sand fraction %)
  ├── Feature 13: Distance to river (m)
  ├── Feature 14: Distance to drainage (m)
  └── Feature 15: HAND — Height Above Nearest Drainage (m) NEW

  Export: soil_distance_hand_amuwo_odofin.tif
  Replaces: soil_distance_amuwo_odofin.tif (delete old file)

  Running total: 15 of 16 features complete
  Remaining: GPM IMERG event rainfall (NB04b)

  Next: 04b_gpm_imerg_rainfall.ipynb
        → Extract 72-hour cumulative rainfall from GPM IMERG
        → One raster per flood event (2017, 2020, 2021, 2022)
        → Feature 16: event-specific rainfall forcing
